## Препроцессинг тестового датасета в единый формат с train_super
Скрипт загружает сырой тестовый файл `test.csv`, применяет к столбцу `text` тот же продвинутый препроцессинг, что и для обучающего датасета (очистка, укорочение длинных текстов), а затем при наличии добавляет доменные токены `[SRC_xxx]` и сохраняет результат в `test_super.csv`. При этом структура теста (включая `ID`) полностью сохраняется, без аугментаций и отборов по количеству, чтобы модели могли корректно делать предсказания для соревнования.


In [ ]:
import os
import pandas as pd

from preprocess import clean_text, shorten_text_by_sentences, MAX_TEXT_LEN

RAW_TEST_PATH   = "../data/ТОНАЛЬНОСТЬ/test.csv"        # сырой тест, выданный на хакатоне
TEST_SUPER_PATH = "../data/processed/test_super.csv"    # обработанный тест для моделей


def add_src_token_if_exists(df: pd.DataFrame) -> pd.DataFrame:
    """
    Добавляем доменный токен [SRC_xxx] в начало текста,
    если в датафрейме есть колонка 'src'.
    """
    df = df.copy()

    if "src" not in df.columns:
        print("⚠️ В test.csv нет колонки 'src' — доменные токены не добавляем.")
        return df

    def add_src_token(row):
        src = str(row["src"]).strip().lower()
        if not src:
            return row["text"]
        token = "[SRC_" + src.replace(" ", "_") + "]"
        return f"{token} {row['text']}"

    df["text"] = df.apply(add_src_token, axis=1)
    print("Добавлены доменные токены [SRC_xxx] в начало текста (test).")
    return df


def preprocess_test_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Полноценный препроцессинг test:
      - text -> str, заполняем NaN
      - clean_text (как в train_super)
      - укорачивание длинных текстов shorten_text_by_sentences
      - добавление [SRC_xxx] (если есть src)
    НИЧЕГО НЕ РЕЖЕМ по количеству, НЕ делаем аугментаций, НЕ меняем ID.
    """
    df = df.copy()

    if "text" not in df.columns:
        raise ValueError("В test.csv нет колонки 'text'")

    # приводим к строкам, NaN -> ""
    df["text"] = df["text"].fillna("").astype(str)

    # очистка (та же, что для train)
    print("Очистка текста (clean_text) для test...")
    df["text"] = df["text"].apply(clean_text)

    # текст может стать пустым — для теста строки НЕ выбрасываем,
    # просто логируем, чтобы понимать масштаб бедствия
    empty_after_clean = (df["text"].str.len() == 0).sum()
    if empty_after_clean > 0:
        print(f"⚠️ Внимание: {empty_after_clean} строк стали пустыми после очистки. "
              "Мы их оставляем, но модель может предсказывать по ним рандом.")

    # укорачиваем длинные тексты
    if MAX_TEXT_LEN is not None:
        long_cnt = (df["text"].str.len() > MAX_TEXT_LEN).sum()
        print(f"Длинных текстов (> {MAX_TEXT_LEN} символов) до укорочения: {long_cnt}")
        if long_cnt > 0:
            df["text"] = df["text"].apply(
                lambda t: shorten_text_by_sentences(t, max_len=MAX_TEXT_LEN)
            )
            long_cnt_after = (df["text"].str.len() > MAX_TEXT_LEN).sum()
            print(f"Длинных текстов после укорочения (ещё > {MAX_TEXT_LEN}): {long_cnt_after}")

    # доменные токены [SRC_xxx]
    df = add_src_token_if_exists(df)

    # статистика
    lengths = df["text"].str.len()
    print("\nСтатистика длины текста (символы) в test ПОСЛЕ препроцессинга:")
    print(lengths.describe(percentiles=[0.1, 0.5, 0.9, 0.95, 0.99]))

    return df


def main():
    if not os.path.exists(RAW_TEST_PATH):
        raise FileNotFoundError(f"Не найден {RAW_TEST_PATH}")

    print(f"Читаем сырой test из {RAW_TEST_PATH}...")
    test_df_raw = pd.read_csv(RAW_TEST_PATH)
    print("Форма сырого test:", test_df_raw.shape)
    print("Колонки:", list(test_df_raw.columns))

    if "ID" not in test_df_raw.columns:
        print("⚠️ В test.csv нет колонки 'ID'. Если она нужна для сабмита, проверь формат файла.")

    test_df_proc = preprocess_test_df(test_df_raw)

    os.makedirs(os.path.dirname(TEST_SUPER_PATH), exist_ok=True)
    test_df_proc.to_csv(TEST_SUPER_PATH, index=False)
    print(f"\n✅ Обработанный test сохранён в {TEST_SUPER_PATH}")
    print(f"Всего строк в обработанном тесте: {len(test_df_proc)}")


if __name__ == "__main__":
    main()


Читаем сырой test из ../data/ТОНАЛЬНОСТЬ/test.csv...
Форма сырого test: (58092, 3)
Колонки: ['ID', 'text', 'src']
Очистка текста (clean_text) для test...
Длинных текстов (> 1200 символов) до укорочения: 2931
Длинных текстов после укорочения (ещё > 1200): 0
Добавлены доменные токены [SRC_xxx] в начало текста (test).

Статистика длины текста (символы) в test ПОСЛЕ препроцессинга:
count    58092.000000
mean       310.824503
std        295.681762
min         15.000000
10%         54.000000
50%        189.000000
90%        798.000000
95%        986.000000
99%       1168.000000
max       1216.000000
Name: text, dtype: float64

✅ Обработанный test сохранён в ../data/processed/test_super.csv
Всего строк в обработанном тесте: 58092
